In [74]:
import pandas as pd

df_results = pd.read_json('serp_fc_results.jsonl', lines=True)

In [ ]:
# Add content length columns

from pathlib import Path
BASE = Path("../samples/ymyl_29000/res_20250723_n100").resolve()
SCRAPED_CSV = BASE / "_scraped.csv"

df_scraped = pd.read_csv(SCRAPED_CSV)

df_results = df_results.merge(df_scraped[['url', 'content']], on='url', how='left')

df_results['content_char_count'] = df_results['content'].fillna('').str.len()
df_results['content_word_count'] = df_results['content'].fillna('').str.split().str.len()
df_results['content_sentence_count'] = df_results['content'].fillna('').str.count(r'[.!?]') + 1

In [ ]:
# Count T/F claims for each document

df_results['true_count'] = df_results['checker_response'].apply(
    lambda x: sum(r.get('classification') == 'True'
                  for r in (x.get('data', {}).get('results', []) if isinstance(x, dict) else []))
)

df_results['false_count'] = df_results['checker_response'].apply(
    lambda x: sum(r.get('classification') == 'False'
                  for r in (x.get('data', {}).get('results', []) if isinstance(x, dict) else []))
)

In [ ]:
ai_results = df_results[df_results['ai_class'] == 'AI']
human_results = df_results[df_results['ai_class'] == 'Human']

ai_count = len(ai_results)
human_count = len(human_results)
total_count = len(df_results)

print('AI count:\t', ai_count)
print('Human count:\t', human_count)
print('---')
print('Total count:\t', total_count)

print()
print('AI share:\t', f'{(ai_count / total_count) * 100:.4f}%')

AI count:	 1735
Human count:	 15345
---
Total count:	 17080

AI share:	 10.1581%


In [ ]:
# Group by ai_class and aggregate counts
fact_stats = (
    df_results.groupby('ai_class')[['true_count', 'false_count']]
    .agg(['sum', 'mean', 'median'])
)

print(fact_stats)

         true_count                   false_count                 
                sum       mean median         sum      mean median
ai_class                                                          
AI            23153  13.344669   13.0        1921  1.107205    1.0
Human        159699  10.407234   10.0       18707  1.219094    1.0
